# CatBoost Direct Horizon Baseline

This notebook implements the fixed-horizon setup for predicting road condition `h` years ahead, where `h ∈ {1, 2, 3, 4}`.

Modeling choices:
- Direct forecast instead of next-measurement prediction
- One shared model per target with `forecast_horizon_years` as an input feature
- Targets are modeled as deltas: `delta_IRI_h` and `delta_URA_h`
- Final predictions are reconstructed as `current + predicted_delta`
- Target lookup stays inside the same lifecycle using the nearest PTM measurement within the configured tolerance window

Recommended pipeline before running this notebook:
1. Run `build_model_dataset_v2.py`
2. Run `build_fixed_horizon_dataset.py`
3. Open this notebook and train/evaluate the models


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


In [3]:
DATA_PATH = Path('data/road_model_dataset_fixed_horizon_v2.parquet')

VALID_START_YEAR = 2017
TEST_START_YEAR = 2018
TEST_END_YEAR = 2020
RANDOM_STATE = 42

FORECAST_YEARS = [1, 2, 3, 4]

BASE_FEATURES = [
    'IRI',
    'URA',
    'Pavement_Age_years',
    'Initial_URA',
    'Measurement_Idx',
    'prev_IRI',
    'prev_URA',
    'Delta_t_years',
    'KVL',
    'KVL_raskas',
    'KVL_kaista',
    'Nopeus',
    'Minor_TP_Count',
    'tp_count_interval',
    'has_TP_interval',
    'Toim_lk',
    'ELY',
    'tp_surface_type',
    'tp_material_type',
    'Tie',
    'Ajorata',
    'Kaista',
    'cycle_num',
]

DERIVED_FEATURES = [
    'heavy_share',
    'iri_rate',
    'ura_rate',
    'age_x_traffic',
    'age_x_heavy',
    'forecast_horizon_years',
]

CATEGORICAL_FEATURES = [
    'Toim_lk',
    'ELY',
    'tp_surface_type',
    'tp_material_type',
]


In [4]:
def require_dataset(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f'Missing dataset: {path}. Run build_fixed_horizon_dataset.py first.'
        )


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    kvl = pd.to_numeric(out['KVL'], errors='coerce')
    kvl_raskas = pd.to_numeric(out['KVL_raskas'], errors='coerce')
    delta_t = pd.to_numeric(out['Delta_t_years'], errors='coerce')

    out['heavy_share'] = np.where(kvl > 0, kvl_raskas / kvl, np.nan)
    out['iri_rate'] = np.where(delta_t > 0, (out['IRI'] - out['prev_IRI']) / delta_t, np.nan)
    out['ura_rate'] = np.where(delta_t > 0, (out['URA'] - out['prev_URA']) / delta_t, np.nan)
    out['age_x_traffic'] = out['Pavement_Age_years'] * kvl
    out['age_x_heavy'] = out['Pavement_Age_years'] * kvl_raskas

    for col in CATEGORICAL_FEATURES:
        out[col] = out[col].astype('string').fillna('missing')

    return out


def stack_horizons(df: pd.DataFrame, forecast_years: list[int]) -> pd.DataFrame:
    pieces = []

    for horizon in forecast_years:
        iri_col = f'target_IRI_{horizon}y'
        ura_col = f'target_URA_{horizon}y'
        actual_col = f'actual_horizon_years_{horizon}y'

        part = df.copy()
        part['forecast_horizon_years'] = horizon
        part['target_IRI'] = part[iri_col]
        part['target_URA'] = part[ura_col]
        part['actual_horizon_years'] = part[actual_col]

        part['delta_IRI'] = part['target_IRI'] - part['IRI']
        part['delta_URA'] = part['target_URA'] - part['URA']

        part['baseline_persist_pred_IRI'] = part['IRI']
        part['baseline_persist_pred_URA'] = part['URA']

        part['baseline_trend_pred_IRI'] = part['IRI'] + part['forecast_horizon_years'] * part['iri_rate']
        part['baseline_trend_pred_URA'] = part['URA'] + part['forecast_horizon_years'] * part['ura_rate']

        part = part[part['target_IRI'].notna() & part['target_URA'].notna()].copy()
        pieces.append(part)

    stacked = pd.concat(pieces, ignore_index=True)
    stacked['event_year'] = pd.to_datetime(stacked['event_date']).dt.year
    return stacked


def regression_metrics(y_true, y_pred, model_name: str, target_name: str, split_name: str, horizon=None):
    mask = pd.Series(y_true).notna() & pd.Series(y_pred).notna()
    y_true = pd.Series(y_true)[mask]
    y_pred = pd.Series(y_pred)[mask]

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        'model': model_name,
        'target': target_name,
        'split': split_name,
        'horizon_years': horizon,
        'rows': int(len(y_true)),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': rmse,
        'r2': r2_score(y_true, y_pred),
    }


def evaluate_predictions(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []

    rows.append(regression_metrics(df['target_IRI'], df['pred_IRI'], 'catboost', 'IRI', split_name))
    rows.append(regression_metrics(df['target_URA'], df['pred_URA'], 'catboost', 'URA', split_name))
    rows.append(regression_metrics(df['target_IRI'], df['baseline_persist_pred_IRI'], 'persistence', 'IRI', split_name))
    rows.append(regression_metrics(df['target_URA'], df['baseline_persist_pred_URA'], 'persistence', 'URA', split_name))
    rows.append(regression_metrics(df['target_IRI'], df['baseline_trend_pred_IRI'], 'trend', 'IRI', split_name))
    rows.append(regression_metrics(df['target_URA'], df['baseline_trend_pred_URA'], 'trend', 'URA', split_name))

    for horizon in FORECAST_YEARS:
        part = df[df['forecast_horizon_years'] == horizon].copy()
        if part.empty:
            continue

        rows.append(regression_metrics(part['target_IRI'], part['pred_IRI'], 'catboost', 'IRI', split_name, horizon))
        rows.append(regression_metrics(part['target_URA'], part['pred_URA'], 'catboost', 'URA', split_name, horizon))
        rows.append(regression_metrics(part['target_IRI'], part['baseline_persist_pred_IRI'], 'persistence', 'IRI', split_name, horizon))
        rows.append(regression_metrics(part['target_URA'], part['baseline_persist_pred_URA'], 'persistence', 'URA', split_name, horizon))
        rows.append(regression_metrics(part['target_IRI'], part['baseline_trend_pred_IRI'], 'trend', 'IRI', split_name, horizon))
        rows.append(regression_metrics(part['target_URA'], part['baseline_trend_pred_URA'], 'trend', 'URA', split_name, horizon))

    return pd.DataFrame(rows)


In [5]:
require_dataset(DATA_PATH)

df = pd.read_parquet(DATA_PATH)
df = add_derived_features(df)
long_df = stack_horizons(df, FORECAST_YEARS)

print(f'Loaded fixed-horizon rows: {len(df):,}')
print(f'Long-format training rows: {len(long_df):,}')
print(f'Event years: {int(long_df.event_year.min())} - {int(long_df.event_year.max())}')
print(long_df[['forecast_horizon_years', 'target_IRI', 'target_URA', 'actual_horizon_years']].head())


Loaded fixed-horizon rows: 3,833,323
Long-format training rows: 4,190,943
Event years: 2007 - 2022
   forecast_horizon_years  target_IRI  target_URA  actual_horizon_years
0                       1        3.80        11.8              0.876112
1                       1        1.60         1.6              0.876112
2                       1        1.37         2.3              1.188227
3                       1        2.20         9.1              0.876112
4                       1        2.12        13.3              1.188227


## Time-based split

This notebook uses a chronological split instead of a random row split.

Default policy:
- train: `event_year < VALID_START_YEAR`
- validation: `VALID_START_YEAR <= event_year < TEST_START_YEAR`
- test: `TEST_START_YEAR <= event_year < TEST_END_YEAR`

This default keeps all forecast horizons 1-4 available in both validation and test.
Adjust the year cut points above if you want a different operational evaluation window.


In [6]:
train_df = long_df[long_df['event_year'] < VALID_START_YEAR].copy()
valid_df = long_df[(long_df['event_year'] >= VALID_START_YEAR) & (long_df['event_year'] < TEST_START_YEAR)].copy()
test_df = long_df[(long_df['event_year'] >= TEST_START_YEAR) & (long_df['event_year'] < TEST_END_YEAR)].copy()

print(f'train rows: {len(train_df):,}')
print(f'valid rows: {len(valid_df):,}')
print(f'test rows:  {len(test_df):,}')

for name, part in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
    print(f'\n{name} horizon counts:')
    print(part['forecast_horizon_years'].value_counts().sort_index())


train rows: 2,648,786
valid rows: 379,132
test rows:  684,586

train horizon counts:
forecast_horizon_years
1    881925
2    688697
3    601943
4    476221
Name: count, dtype: int64

valid horizon counts:
forecast_horizon_years
1     57214
2     68216
3    165416
4     88286
Name: count, dtype: int64

test horizon counts:
forecast_horizon_years
1    175464
2    153315
3    258010
4     97797
Name: count, dtype: int64


In [7]:
MODEL_FEATURES = BASE_FEATURES + DERIVED_FEATURES
cat_feature_indices = [MODEL_FEATURES.index(col) for col in CATEGORICAL_FEATURES]

X_train = train_df[MODEL_FEATURES]
X_valid = valid_df[MODEL_FEATURES]
X_test = test_df[MODEL_FEATURES]

iri_model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=RANDOM_STATE,
    iterations=800,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5.0,
    min_data_in_leaf=100,
    verbose=100,
)

ura_model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=RANDOM_STATE,
    iterations=800,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5.0,
    min_data_in_leaf=100,
    verbose=100,
)


In [ ]:
iri_model.fit(
    X_train,
    train_df['delta_IRI'],
    cat_features=cat_feature_indices,
    eval_set=(X_valid, valid_df['delta_IRI']),
    use_best_model=True,
)

ura_model.fit(
    X_train,
    train_df['delta_URA'],
    cat_features=cat_feature_indices,
    eval_set=(X_valid, valid_df['delta_URA']),
    use_best_model=True,
)


In [8]:
for part in [train_df, valid_df, test_df]:
    part['pred_delta_IRI'] = iri_model.predict(part[MODEL_FEATURES])
    part['pred_delta_URA'] = ura_model.predict(part[MODEL_FEATURES])
    part['pred_IRI'] = part['IRI'] + part['pred_delta_IRI']
    part['pred_URA'] = part['URA'] + part['pred_delta_URA']


CatBoostError: There is no trained model to use predict(). Use fit() to train model. Then use this method.

In [ ]:
metrics_df = pd.concat(
    [
        evaluate_predictions(train_df, 'train'),
        evaluate_predictions(valid_df, 'valid'),
        evaluate_predictions(test_df, 'test'),
    ],
    ignore_index=True,
)

metrics_df.sort_values(['split', 'target', 'horizon_years', 'model']).head(30)


In [ ]:
summary_overall = metrics_df[metrics_df['horizon_years'].isna()].copy()
summary_overall.sort_values(['split', 'target', 'mae'])


In [ ]:
summary_by_horizon = metrics_df[metrics_df['horizon_years'].notna()].copy()
summary_by_horizon.sort_values(['split', 'target', 'horizon_years', 'mae'])


In [ ]:
def breakdown_mae(df: pd.DataFrame, group_col: str, split_name: str) -> pd.DataFrame:
    rows = []
    for target_name, pred_col, true_col in [
        ('IRI', 'pred_IRI', 'target_IRI'),
        ('URA', 'pred_URA', 'target_URA'),
    ]:
        grouped = df.groupby([group_col, 'forecast_horizon_years'])
        for (group_value, horizon), part in grouped:
            rows.append(
                {
                    'split': split_name,
                    'group_col': group_col,
                    'group_value': group_value,
                    'target': target_name,
                    'horizon_years': horizon,
                    'rows': len(part),
                    'mae': mean_absolute_error(part[true_col], part[pred_col]),
                }
            )
    return pd.DataFrame(rows)

ely_breakdown = breakdown_mae(test_df, 'ELY', 'test')
road_class_breakdown = breakdown_mae(test_df, 'Toim_lk', 'test')

ely_breakdown.sort_values(['target', 'horizon_years', 'mae']).head(20)


In [ ]:
road_class_breakdown.sort_values(['target', 'horizon_years', 'mae']).head(20)


In [ ]:
importance_iri = pd.DataFrame(
    {
        'feature': MODEL_FEATURES,
        'importance': iri_model.get_feature_importance(),
    }
).sort_values('importance', ascending=False)

importance_ura = pd.DataFrame(
    {
        'feature': MODEL_FEATURES,
        'importance': ura_model.get_feature_importance(),
    }
).sort_values('importance', ascending=False)

importance_iri.head(20), importance_ura.head(20)


## Horizon Summary

This summary compares CatBoost against the persistence baseline separately for each forecast horizon and target on the validation and test splits.


In [9]:
horizon_summary = (
    summary_by_horizon[summary_by_horizon['model'].isin(['catboost', 'persistence'])]
    .pivot_table(
        index=['split', 'target', 'horizon_years'],
        columns='model',
        values=['rows', 'mae', 'rmse', 'r2'],
    )
)

horizon_summary.columns = [f'{metric}_{model}' for metric, model in horizon_summary.columns]
horizon_summary = horizon_summary.reset_index()

for metric in ['mae', 'rmse', 'r2']:
    horizon_summary[f'{metric}_gain_vs_persistence'] = (
        horizon_summary[f'{metric}_persistence'] - horizon_summary[f'{metric}_catboost']
    )

horizon_summary['r2_gain_vs_persistence'] = (
    horizon_summary['r2_catboost'] - horizon_summary['r2_persistence']
)

horizon_summary['rmse_reduction_pct_vs_persistence'] = (
    100.0
    * horizon_summary['rmse_gain_vs_persistence']
    / horizon_summary['rmse_persistence']
)

horizon_summary['mae_reduction_pct_vs_persistence'] = (
    100.0
    * horizon_summary['mae_gain_vs_persistence']
    / horizon_summary['mae_persistence']
)

horizon_summary = horizon_summary.sort_values(['split', 'target', 'horizon_years'])

cols_to_show = [
    'split',
    'target',
    'horizon_years',
    'rows_catboost',
    'mae_catboost',
    'rmse_catboost',
    'r2_catboost',
    'mae_persistence',
    'rmse_persistence',
    'r2_persistence',
    'mae_reduction_pct_vs_persistence',
    'rmse_reduction_pct_vs_persistence',
    'r2_gain_vs_persistence',
]

horizon_summary[cols_to_show].round(4)


NameError: name 'summary_by_horizon' is not defined

## Using Saved Models

Use the cells below when you want to load already trained CatBoost models from disk and run predictions for a chosen forecast horizon without retraining.


In [10]:
MODEL_DIR = Path('models')
IRI_MODEL_PATH = MODEL_DIR / 'catboost_iri.cbm'
URA_MODEL_PATH = MODEL_DIR / 'catboost_ura.cbm'

loaded_iri_model = CatBoostRegressor()
loaded_ura_model = CatBoostRegressor()

loaded_iri_model.load_model(IRI_MODEL_PATH)
loaded_ura_model.load_model(URA_MODEL_PATH)

print(f'Loaded IRI model: {IRI_MODEL_PATH}')
print(f'Loaded URA model: {URA_MODEL_PATH}')


Loaded IRI model: models/catboost_iri.cbm
Loaded URA model: models/catboost_ura.cbm


In [11]:
PREDICT_PATH = DATA_PATH
PREDICT_HORIZON_YEARS = 1
PREDICT_SAMPLE_ROWS = 10

predict_df = pd.read_parquet(PREDICT_PATH)
predict_df = add_derived_features(predict_df)
predict_df['forecast_horizon_years'] = PREDICT_HORIZON_YEARS

X_pred = predict_df[MODEL_FEATURES]

predict_df['pred_delta_IRI'] = loaded_iri_model.predict(X_pred)
predict_df['pred_delta_URA'] = loaded_ura_model.predict(X_pred)
predict_df['pred_IRI'] = predict_df['IRI'] + predict_df['pred_delta_IRI']
predict_df['pred_URA'] = predict_df['URA'] + predict_df['pred_delta_URA']

prediction_columns = [
    'Lifecycle_ID',
    'event_date',
    'forecast_horizon_years',
    'IRI',
    'URA',
    'pred_delta_IRI',
    'pred_delta_URA',
    'pred_IRI',
    'pred_URA',
]

predict_df[prediction_columns].head(PREDICT_SAMPLE_ROWS)


,Lifecycle_ID,event_date,forecast_horizon_years,IRI,URA,pred_delta_IRI,pred_delta_URA,pred_IRI,pred_URA
0,Epo_13309_0_11_3_6900_3_7000_C0,2018-06-18,1,1.60,2.7,0.072943,0.335652,1.672943,3.035652
1,Epo_13309_0_11_3_7000_3_7100_C0,2018-06-18,1,1.50,1.4,0.079111,0.274358,1.579111,1.674358
2,Epo_13309_0_11_3_7100_3_7200_C0,2018-06-18,1,1.75,1.5,0.064301,0.332280,1.814301,1.832280
3,Epo_13309_0_11_3_7200_3_7300_C0,2018-06-18,1,1.45,4.7,0.086737,0.176958,1.536737,4.876958
4,Epo_13309_0_11_3_7300_3_7400_C0,2018-06-18,1,2.42,1.4,-0.002491,0.447715,2.417509,1.847715
5,Epo_13309_0_21_3_6900_3_7000_C0,2018-06-16,1,2.93,3.5,-0.075888,0.430128,2.854112,3.930128
6,Epo_13309_0_21_3_7000_3_7100_C0,2018-06-16,1,1.82,1.4,0.045399,0.329118,1.865399,1.729118
7,Epo_13309_0_21_3_7100_3_7200_C0,2018-06-16,1,2.02,0.9,0.039289,0.327432,2.059289,1.227432
8,Epo_13309_0_21_3_7200_3_7300_C0,2018-06-16,1,1.44,2.3,0.063031,0.274594,1.503031,2.574594
9,Epo_13309_0_21_3_7300_3_7400_C0,2018-06-16,1,2.07,4.5,0.056032,0.302036,2.126032,4.802036


In [12]:
PREDICTION_OUTPUT_PATH = MODEL_DIR / f'predictions_{PREDICT_HORIZON_YEARS}y.parquet'

predict_df[prediction_columns].to_parquet(PREDICTION_OUTPUT_PATH, index=False)
print(f'Saved predictions: {PREDICTION_OUTPUT_PATH}')


Saved predictions: models/predictions_1y.parquet


## Accuracy From Saved Models

Run this cell if you want to rebuild validation and test metrics from the saved `.cbm` files without calling `fit(...)` again.


In [17]:
require_dataset(DATA_PATH)

metrics_source_df = pd.read_parquet(DATA_PATH)
metrics_source_df = add_derived_features(metrics_source_df)
metrics_long_df = stack_horizons(metrics_source_df, FORECAST_YEARS)

metrics_valid_df = metrics_long_df[
    (metrics_long_df['event_year'] >= VALID_START_YEAR)
    & (metrics_long_df['event_year'] < TEST_START_YEAR)
].copy()
metrics_test_df = metrics_long_df[
    (metrics_long_df['event_year'] >= TEST_START_YEAR)
    & (metrics_long_df['event_year'] < TEST_END_YEAR)
].copy()

loaded_iri_model = CatBoostRegressor()
loaded_ura_model = CatBoostRegressor()
loaded_iri_model.load_model(IRI_MODEL_PATH)
loaded_ura_model.load_model(URA_MODEL_PATH)

for part in [metrics_valid_df, metrics_test_df]:
    part['pred_delta_IRI'] = loaded_iri_model.predict(part[MODEL_FEATURES])
    part['pred_delta_URA'] = loaded_ura_model.predict(part[MODEL_FEATURES])
    part['pred_IRI'] = part['IRI'] + part['pred_delta_IRI']
    part['pred_URA'] = part['URA'] + part['pred_delta_URA']

metrics_from_saved_models = pd.concat(
    [
        evaluate_predictions(metrics_valid_df, 'valid'),
        evaluate_predictions(metrics_test_df, 'test'),
    ],
    ignore_index=True,
)

saved_model_accuracy_summary = metrics_from_saved_models[
    metrics_from_saved_models['model'] == 'catboost'
].copy()
saved_model_accuracy_summary['level'] = np.where(
    saved_model_accuracy_summary['horizon_years'].isna(), 'overall', 'by_horizon'
)

saved_model_accuracy_summary = saved_model_accuracy_summary[
    ['level', 'split', 'target', 'horizon_years', 'rows', 'mae', 'rmse', 'r2']
].sort_values(['level', 'split', 'target', 'horizon_years'])

saved_model_vs_persistence = metrics_from_saved_models[
    metrics_from_saved_models['model'].isin(['catboost', 'persistence'])
].copy()
saved_model_vs_persistence['level'] = np.where(
    saved_model_vs_persistence['horizon_years'].isna(), 'overall', 'by_horizon'
)

saved_model_vs_persistence = saved_model_vs_persistence.pivot_table(
    index=['level', 'split', 'target', 'horizon_years'],
    columns='model',
    values=['rows', 'mae', 'rmse', 'r2'],
)

saved_model_vs_persistence.columns = [
    f'{metric}_{model}' for metric, model in saved_model_vs_persistence.columns
]
saved_model_vs_persistence = saved_model_vs_persistence.reset_index()

saved_model_vs_persistence['mae_reduction_pct_vs_persistence'] = (
    100.0
    * (saved_model_vs_persistence['mae_persistence'] - saved_model_vs_persistence['mae_catboost'])
    / saved_model_vs_persistence['mae_persistence']
)
saved_model_vs_persistence['rmse_reduction_pct_vs_persistence'] = (
    100.0
    * (saved_model_vs_persistence['rmse_persistence'] - saved_model_vs_persistence['rmse_catboost'])
    / saved_model_vs_persistence['rmse_persistence']
)
saved_model_vs_persistence['r2_gain_vs_persistence'] = (
    saved_model_vs_persistence['r2_catboost'] - saved_model_vs_persistence['r2_persistence']
)

saved_model_vs_persistence = saved_model_vs_persistence[
    [
        'level',
        'split',
        'target',
        'horizon_years',
        'rows_catboost',
        'mae_catboost',
        'rmse_catboost',
        'r2_catboost',
        'mae_persistence',
        'rmse_persistence',
        'r2_persistence',
        'mae_reduction_pct_vs_persistence',
        'rmse_reduction_pct_vs_persistence',
        'r2_gain_vs_persistence',
    ]
].sort_values(['level', 'split', 'target', 'horizon_years'])

saved_model_accuracy_summary.round(4), saved_model_vs_persistence.round(4)


(         level  split target  horizon_years    rows     mae    rmse      r2
 36  by_horizon   test    IRI            1.0  175464  0.1303  0.2598  0.7685
 42  by_horizon   test    IRI            2.0  153315  0.1496  0.2685  0.8344
 48  by_horizon   test    IRI            3.0  258010  0.1941  0.3323  0.8524
 54  by_horizon   test    IRI            4.0   97797  0.1841  0.3287  0.8038
 37  by_horizon   test    URA            1.0  175464  0.7130  1.0134  0.9266
 43  by_horizon   test    URA            2.0  153315  0.9152  1.2744  0.8893
 49  by_horizon   test    URA            3.0  258010  1.0957  1.5810  0.8351
 55  by_horizon   test    URA            4.0   97797  1.3139  1.8305  0.8110
 6   by_horizon  valid    IRI            1.0   57214  0.1370  0.3039  0.6926
 12  by_horizon  valid    IRI            2.0   68216  0.1436  0.2956  0.7512
 18  by_horizon  valid    IRI            3.0  165416  0.2434  0.4016  0.8367
 24  by_horizon  valid    IRI            4.0   88286  0.2670  0.4403  0.8702

## Notes

Current target construction uses the nearest observed PTM inside the same lifecycle and within the tolerance window defined in `build_fixed_horizon_dataset.py`.

This means:
- the setup is already a direct-horizon problem, not a next-measurement problem
- major reset behavior is still implicitly controlled by lifecycle boundaries
- interpolation is not yet applied; if you want tighter target semantics, the next step is to replace nearest-observation lookup with forward interpolation between future PTM observations


In [16]:
model_accuracy_summary = metrics_df.copy()
model_accuracy_summary['level'] = np.where(
    model_accuracy_summary['horizon_years'].isna(), 'overall', 'by_horizon'
)

model_accuracy_summary = model_accuracy_summary[
    (model_accuracy_summary['model'] == 'catboost')
    & (model_accuracy_summary['split'].isin(['valid', 'test']))
].copy()

model_accuracy_summary = model_accuracy_summary[
    ['level', 'split', 'target', 'horizon_years', 'rows', 'mae', 'rmse', 'r2']
].sort_values(['level', 'split', 'target', 'horizon_years'])

model_accuracy_summary.round(4)


NameError: name 'metrics_df' is not defined

In [15]:
required_names = [
    'long_df',
    'train_df',
    'valid_df',
    'test_df',
    'metrics_df',
]

available_names = set(globals())
missing_names = [name for name in required_names if name not in available_names]

if not missing_names:
    print('Notebook state OK: metrics and split data are already in memory.')
    print('You do not need to retrain the model to view the summary cells.')
else:
    print('Missing notebook variables:', ', '.join(missing_names))
    print('If models are already saved, you do not need to retrain.')
    print('Load the saved models, rebuild predictions, and recompute metrics_df.')


Missing notebook variables: metrics_df
If models are already saved, you do not need to retrain.
Load the saved models, rebuild predictions, and recompute metrics_df.
